# LINZ AWS Data Access with obstore

This notebook demonstrates how to access LINZ public datasets hosted on AWS S3 using the obstore library.

**Features covered:**
- Setting up AWS S3 access for public LINZ datasets
- Listing objects in S3 buckets
- Downloading single files
- Batch downloading multiple images
- Reading file headers without full download

## Prerequisites

Install required dependencies:
```bash
pip install obstore
```

## 1. Import Dependencies and Setup

In [ ]:
from __future__ import annotations

import argparse
from dataclasses import dataclass
from pathlib import Path
import time
import os

import obstore as obs
from obstore.store import S3Store

## 2. Define Dataset Configuration

In [ ]:
@dataclass(frozen=True)
class DatasetInfo:
    bucket: str
    region: str

# Available LINZ datasets
LINZ_DATASETS: dict[str, DatasetInfo] = {
    "imagery": DatasetInfo(bucket="nz-imagery", region="ap-southeast-2"),
    "elevation": DatasetInfo(bucket="nz-elevation", region="ap-southeast-2"),
    "coastal": DatasetInfo(bucket="nz-coastal", region="ap-southeast-2"),
}

print("Available LINZ datasets:")
for name, info in LINZ_DATASETS.items():
    print(f"  {name}: {info.bucket} ({info.region})")

## 3. Core Functions for AWS S3 Access

In [ ]:
def get_public_store(bucket: str, region: str) -> S3Store:
    """Create an unsigned S3 store for public AWS Open Data buckets."""
    return S3Store(
        bucket=bucket,
        region=region,
        skip_signature=True,
    )

def download_object(store: S3Store, path: str, output_file: str) -> int:
    """Download a full object from S3 to a local file.
    
    Returns the number of bytes written.
    """
    response = obs.get(store, path)
    data = response.bytes()
    with open(output_file, "wb") as file_handle:
        file_handle.write(data)
    return len(data)

def read_object_range(store: S3Store, path: str, length: int) -> bytes:
    """Read a byte range from the start of an object (useful for COG headers)."""
    buffer = obs.get_range(store, path, start=0, length=length)
    return bytes(buffer)

def list_objects(
    store: S3Store,
    prefix: str,
    *,
    endswith: str = "",
    limit: int = 0,
) -> list[str]:
    """List object paths under a prefix, optionally filtering by suffix and limit."""
    paths: list[str] = []
    stream = obs.list(store, prefix=prefix)

    for chunk in stream:
        for item in chunk:
            path = item["path"]
            if endswith and not path.endswith(endswith):
                continue
            paths.append(path)
            if limit > 0 and len(paths) >= limit:
                return paths

    return paths

print("✅ Core functions defined successfully!")

## 4. Example: Set Up Connection to LINZ Imagery Dataset

In [ ]:
# Choose a dataset
dataset_name = "imagery"
dataset = LINZ_DATASETS[dataset_name]

# Create the S3 store
store = get_public_store(bucket=dataset.bucket, region=dataset.region)

print(f"Connected to {dataset_name} dataset:")
print(f"  Bucket: {dataset.bucket}")
print(f"  Region: {dataset.region}")

## 5. List Objects in a Directory

In [ ]:
# Example: List files in Taranaki directory
prefix = "taranaki/taranaki_2022-2023_0.1m/rgb/2193/"

print(f"Listing objects under prefix: {prefix}")
print("Searching for TIFF files...")

# List TIFF files with a limit
tiff_files = list_objects(
    store=store,
    prefix=prefix,
    endswith=".tiff",
    limit=10  # Limit to first 10 files
)

if tiff_files:
    print(f"\nFound {len(tiff_files)} TIFF files:")
    for i, file_path in enumerate(tiff_files, 1):
        filename = file_path.split('/')[-1]
        print(f"  {i}. {filename}")
else:
    print("No TIFF files found.")

## 6. Download a Single File

In [ ]:
# Download the first file from our list (if available)
if tiff_files:
    # Take the first file
    target_file = tiff_files[0]
    filename = target_file.split('/')[-1]
    output_path = f"sample_{filename}"
    
    print(f"Downloading: {filename}")
    print(f"From: {target_file}")
    
    start_time = time.time()
    size = download_object(store=store, path=target_file, output_file=output_path)
    elapsed = time.time() - start_time
    
    print(f"✅ Downloaded {size / (1024 * 1024):.2f} MB in {elapsed:.2f} seconds")
    print(f"💾 Saved as: {output_path}")
else:
    print("No files available to download. Try a different prefix.")

## 7. Read File Headers Only (Without Full Download)

In [ ]:
# Read just the first few bytes of a file (useful for checking file types)
if tiff_files:
    target_file = tiff_files[0]
    header_length = 2048  # Read first 2KB
    
    print(f"Reading {header_length} bytes from: {target_file.split('/')[-1]}")
    
    header_data = read_object_range(store=store, path=target_file, length=header_length)
    
    print(f"✅ Read {len(header_data)} bytes")
    print(f"📄 First 32 bytes (hex): {header_data[:32].hex()}")
    
    # Check if it's a TIFF file
    if header_data.startswith(b'II*\x00') or header_data.startswith(b'MM\x00*'):
        print("🔍 Confirmed: This is a valid TIFF file!")
    else:
        print("❓ This doesn't appear to be a standard TIFF file.")

## 8. Batch Download Function

In [ ]:
def download_all_images(
    store: S3Store,
    prefix: str,
    output_dir: str = "downloads",
    *,
    image_extensions: tuple[str, ...] = (".tif", ".tiff", ".jpg", ".jpeg", ".png"),
    limit: int = 0,
) -> list[str]:
    """Download all image files from a given S3 prefix to a local directory.
    
    Args:
        store: S3Store instance
        prefix: S3 path prefix to search for images
        output_dir: Local directory to save downloaded images
        image_extensions: File extensions to consider as images
        limit: Maximum number of images to download (0 = no limit)
        
    Returns:
        List of successfully downloaded file paths
    """
    # Ensure output directory exists
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    # Get list of image files
    print(f"🔍 Scanning for images in: {prefix}")
    all_paths = list_objects(store=store, prefix=prefix, limit=limit)
    
    # Filter for image files
    image_paths = [
        path for path in all_paths 
        if any(path.lower().endswith(ext) for ext in image_extensions)
    ]
    
    if not image_paths:
        print("❌ No image files found.")
        return []
    
    print(f"📁 Found {len(image_paths)} image files to download")
    
    downloaded_files = []
    failed_downloads = []
    
    for i, image_path in enumerate(image_paths, 1):
        try:
            # Extract filename from S3 path
            filename = os.path.basename(image_path)
            output_path = os.path.join(output_dir, filename)
            
            # Skip if file already exists
            if os.path.exists(output_path):
                print(f"[{i}/{len(image_paths)}] ⏭️  Skipping existing file: {filename}")
                downloaded_files.append(output_path)
                continue
            
            print(f"[{i}/{len(image_paths)}] ⬇️  Downloading: {filename}")
            start_time = time.time()
            
            size = download_object(store=store, path=image_path, output_file=output_path)
            
            elapsed = time.time() - start_time
            size_mb = size / (1024 * 1024)
            speed_mbps = size_mb / elapsed if elapsed > 0 else 0
            
            print(f"     ✅ Downloaded {size_mb:.2f} MB in {elapsed:.1f}s ({speed_mbps:.2f} MB/s)")
            downloaded_files.append(output_path)
            
        except Exception as e:
            print(f"     ❌ Failed to download {image_path}: {e}")
            failed_downloads.append(image_path)
    
    print(f"\n📊 Download Summary:")
    print(f"     ✅ Successfully downloaded: {len(downloaded_files)} files")
    print(f"     ❌ Failed downloads: {len(failed_downloads)} files")
    
    if failed_downloads:
        print(f"     Failed files:")
        for failed_path in failed_downloads:
            print(f"       - {failed_path}")
    
    return downloaded_files

print("✅ Batch download function ready!")

## 9. Example: Batch Download Images

In [ ]:
# Download a few images for demonstration
download_prefix = "taranaki/taranaki_2022-2023_0.1m/rgb/2193/"
output_directory = "sample_downloads"

print(f"🚀 Starting batch download from: {download_prefix}")
print(f"📂 Output directory: {output_directory}")

downloaded_files = download_all_images(
    store=store,
    prefix=download_prefix,
    output_dir=output_directory,
    image_extensions=(".tif", ".tiff"),  # Only TIFF files
    limit=3  # Limit to 3 files for demo
)

if downloaded_files:
    print(f"\n🎉 Successfully downloaded {len(downloaded_files)} files:")
    for file_path in downloaded_files:
        file_size = os.path.getsize(file_path) / (1024 * 1024)
        print(f"   📄 {os.path.basename(file_path)} ({file_size:.2f} MB)")
else:
    print("\n❌ No files were downloaded.")

## 10. Convenience Function for Easy Use

In [ ]:
def download_dataset_images(
    dataset_name: str = "imagery",
    path_prefix: str = "taranaki/taranaki_2022-2023_0.1m/rgb/2193/",
    output_dir: str = "downloads",
    limit: int = 0,
) -> list[str]:
    """Convenience function to download all images from a LINZ dataset.
    
    Args:
        dataset_name: One of 'imagery', 'elevation', 'coastal'
        path_prefix: S3 path prefix to search for images
        output_dir: Local directory to save downloaded images  
        limit: Maximum number of images to download (0 = no limit)
        
    Returns:
        List of successfully downloaded file paths
    """
    if dataset_name not in LINZ_DATASETS:
        raise ValueError(f"Unknown dataset: {dataset_name}. Choose from: {list(LINZ_DATASETS.keys())}")
    
    dataset = LINZ_DATASETS[dataset_name]
    store = get_public_store(bucket=dataset.bucket, region=dataset.region)
    
    return download_all_images(
        store=store,
        prefix=path_prefix,
        output_dir=output_dir,
        limit=limit,
    )

# Example usage of convenience function
print("\n🎯 Using convenience function:")
files = download_dataset_images(
    dataset_name="imagery",
    path_prefix="taranaki/taranaki_2022-2023_0.1m/rgb/2193/",
    output_dir="convenience_downloads",
    limit=2
)

print(f"\n✨ Convenience function downloaded {len(files)} files!")

## Summary

This notebook demonstrated how to:

✅ **Connect to LINZ AWS datasets** using obstore  
✅ **List objects** in S3 buckets with filtering  
✅ **Download single files** with progress tracking  
✅ **Read file headers** without full download  
✅ **Batch download multiple images** with error handling  
✅ **Use convenience functions** for common tasks  

### Key Functions:
- `get_public_store()` - Set up S3 connection
- `list_objects()` - List files with filtering
- `download_object()` - Download single file  
- `download_all_images()` - Batch download with progress
- `download_dataset_images()` - One-line convenience function

### Next Steps:
- Try different datasets (elevation, coastal)
- Experiment with different file types and filters
- Integrate with your geospatial analysis workflow
- Check out the rasterio notebook for direct data processing